# 06. Random Forest and HistGradientBoosting (my part)

This is my part of the project. Our team split the work. Samina does the simple average model (the baseline). I do the two tree based models: Random Forest and HistGradientBoosting.

What the models predict: State of Health, or SOH. This is how much life a battery still has, written as a percent. 100 percent means the battery is like new.

The most important rule: I test my models on batteries they have never seen before. That is the real goal of the project. I want to know if the model works on a brand new battery, not only on the batteries we already measured. So the main test (called LOCO) hides one whole battery, trains on the rest, and then checks how well it guesses the hidden one.

Why I keep the models small: I only have 306 measurements from about 15 batteries. That is a small amount of data. So I keep the models simple (shallow trees, and at least 5 data points in each leaf) so they do not just memorize single batteries.

All errors are in SOH percent points. Lower is better. An error of 4 means the guess is about 4 percent off.

In [1]:
# I set up the notebook and import the tools I need.
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

# I hide harmless warnings (like "unknown category") so the notebook stays clean and small.
warnings.filterwarnings('ignore')

# I make sure Python can find the project code whether I run this from the
# notebooks folder or from the main project folder.
ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# I import the two models I am building, plus the team's shared scoring code.
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from src import evaluation as ev

# I create the folder where I will save my results.
OUT = ROOT / 'results' / 'modelling_random_forest_hgb'
(OUT / 'figures').mkdir(parents=True, exist_ok=True)
print('project folder:', ROOT)
print('test methods  :', list(ev.SCHEMES))
print('random seed   :', ev.SEED)

project folder: /Users/kshitijarohandeshmukh/Downloads/Fall2026/Data245_MachineLearning/project soh/battery-soh-prediction
test methods  : ['LOCO', 'LOPO', 'Random 5-fold']
random seed   : 42


## 1. I get the data ready

I load the 306 rows I can label. Each row is one health check up of one battery at 25 degrees. I also load the list of inputs the model is allowed to use. I leave out any column that secretly contains the answer, so the model has to really learn instead of cheating.

In [2]:
# I load the data and the list of allowed inputs, straight from the repo files.
df = ev.load_labelled()
numeric, categorical = ev.main_features()
print(f'rows: {len(df)}   batteries: {df.cell_id.nunique()}   conditions: {df.profile.nunique()}')
print(f'number of inputs: {len(numeric)}   plus category input: {categorical}')
print()
print('how many batteries fall in each health group:')
print(df.health_class.value_counts())
df[['cell_id', 'profile', 'idtest', 'rtp_index', 'SOH', 'health_class']].head()

rows: 306   batteries: 15   conditions: 14
number of inputs: 42   plus category input: ['schedule']

how many batteries fall in each health group:
health_class
Healthy (>=90)      187
Degraded (80-90)     66
EOL risk (<80)       53
Name: count, dtype: int64


,cell_id,profile,idtest,rtp_index,SOH,health_class
0,2_009,L03,3,1,100.000000,Healthy (>=90)
1,2_009,L03,8,2,96.383034,Healthy (>=90)
2,2_009,L03,10,3,95.757077,Healthy (>=90)
3,2_009,L03,12,4,95.419590,Healthy (>=90)
4,2_009,L03,14,5,96.320094,Healthy (>=90)


## 2. I set up my two models

For each model I write a short list of settings to try (called a grid). I write this list before I see any score, and I never change it after. If I changed it after seeing the results, I would just be fixing the numbers to look good, and that is not fair.

My two models:
1. Random Forest. It builds many small decision trees and averages their guesses.
2. HistGradientBoosting. It builds trees one after another, where each new tree fixes the mistakes of the one before.

I keep both models simple on purpose: shallow trees, and every leaf must hold at least 5 data points.

In [3]:
# I pick the settings to try for each model. I wrote these before seeing any result.
SEED = ev.SEED  # 42, so my results are the same every time I run it.

# I build my Random Forest.
rf_spec = ev.ModelSpec(
    name='Random forest',
    estimator=RandomForestRegressor(n_estimators=400, random_state=SEED, n_jobs=1),
    param_grid={
        'max_depth': [4, 8],            # I keep the trees shallow.
        'min_samples_leaf': [5, 10],    # I keep at least 5 points per leaf.
        'max_features': [0.5, 1.0],
    },
    family='non-linear',
)

# I build my HistGradientBoosting.
hgb_spec = ev.ModelSpec(
    name='HistGradientBoosting',
    estimator=HistGradientBoostingRegressor(max_iter=300, random_state=SEED, early_stopping=False),
    param_grid={
        'learning_rate': [0.05, 0.1],
        'max_leaf_nodes': [7, 15],      # I keep the trees shallow.
        'min_samples_leaf': [10, 20],   # I keep at least 5 points per leaf.
    },
    family='non-linear',
)

# I print how many settings each model will try.
for s in (rf_spec, hgb_spec):
    n = int(np.prod([len(v) for v in s.param_grid.values()]))
    print(f'{s.name}: {n} settings to try -> {s.param_grid}')

Random forest: 8 settings to try -> {'max_depth': [4, 8], 'min_samples_leaf': [5, 10], 'max_features': [0.5, 1.0]}
HistGradientBoosting: 8 settings to try -> {'learning_rate': [0.05, 0.1], 'max_leaf_nodes': [7, 15], 'min_samples_leaf': [10, 20]}


## 3. I train and test the models

This cell does the real work. For every model and every test method, it:
1. Hides some batteries.
2. Picks the best settings using only the batteries it is allowed to see.
3. Trains, and then guesses the hidden batteries.

Picking the settings without looking at the hidden batteries is what keeps my score honest. This can take a minute or two to finish.

In [4]:
# I run both models through all three test methods and collect the results.
results = {}        # I store each result here, by (model, test method).
all_results = []    # I also keep a flat list for the summary tables.

for spec in (rf_spec, hgb_spec):
    for scheme in ev.SCHEMES:
        res = ev.run_cv(df, spec, scheme, verbose=True)
        results[(spec.name, scheme)] = res
        all_results.append(res)

print('\ndone. I ran', len(all_results), 'model and test-method combinations.')

Random forest | LOCO: fold 1

Random forest | LOCO: fold 2

Random forest | LOCO: fold 3

Random forest | LOCO: fold 4

Random forest | LOCO: fold 5

Random forest | LOCO: fold 6

... (59 outputs left out)

HistGradientBoosting | Random 5-fold: fold 3

HistGradientBoosting | Random 5-fold: fold 4

HistGradientBoosting | Random 5-fold: fold 5



done. I ran 6 model and test-method combinations.


## 4. My main results

This is the main table. There are three ways of testing, from hardest to easiest:
1. LOCO. I hide one whole battery. This is the number that matters most.
2. LOPO. I hide one whole operating condition.
3. Random 5-fold. This is the easy, too good test, because it lets the model peek at other check ups of the same battery. I show it only to prove how misleading that kind of test looks.

The generalization gap is just how much more optimistic that easy test is compared to the honest one.

In [5]:
# I build my main results table and sort it so the best score is on top for each test.
summary = ev.summary_table(all_results).sort_values(['scheme', 'MAE']).reset_index(drop=True)
summary_round = summary.copy()
for col in ['MAE', 'RMSE', 'R2', 'MaxAE']:
    summary_round[col] = summary_round[col].round(3)
print('My results (MAE and RMSE are in SOH percent points, lower is better):')
display(summary_round)

print()
print('Generalization gap (honest-test error minus easy-test error):')
display(ev.generalization_gap(all_results).round(3))

My results (MAE and RMSE are in SOH percent points, lower is better):


,model,scheme,n,MAE,RMSE,R2,MaxAE
0,HistGradientBoosting,LOCO,306,4.275,7.720,0.516,46.282
1,Random forest,LOCO,306,4.651,8.669,0.390,55.479
2,Random forest,LOPO,306,4.458,8.313,0.439,51.195
3,HistGradientBoosting,LOPO,306,4.490,8.601,0.400,51.011
4,HistGradientBoosting,Random 5-fold,306,1.625,3.316,0.911,27.901
5,Random forest,Random 5-fold,306,2.046,4.512,0.835,42.753



Generalization gap (honest-test error minus easy-test error):


scheme,LOCO,LOPO,Random 5-fold,gap LOCO - random,gap LOPO - random
model,,,,,
HistGradientBoosting,4.275,4.490,1.625,2.650,2.865
Random forest,4.651,4.458,2.046,2.605,2.411


## 5. Where does my model make mistakes?

A single average error hides a lot of the story. So I split the error up:
1. By health group: healthy, getting worse, or near the end of life.
2. By condition: batteries run in conditions I have seen before, versus brand new conditions.
3. By each battery.

This shows honestly when my model can be trusted and when it cannot.

In [6]:
# I pick my better model on the main test (LOCO), then look at where it makes mistakes.
rf_loco_mae = results[('Random forest', 'LOCO')].metrics['MAE']
hgb_loco_mae = results[('HistGradientBoosting', 'LOCO')].metrics['MAE']
best_name = 'Random forest' if rf_loco_mae <= hgb_loco_mae else 'HistGradientBoosting'
best_spec = rf_spec if best_name == 'Random forest' else hgb_spec
print(f'My better model on the main test: {best_name} '
      f'(Random forest={rf_loco_mae:.3f}, HistGradientBoosting={hgb_loco_mae:.3f})')

oof_best = results[(best_name, 'LOCO')].oof
print()
print('Error by health group:')
display(ev.breakdown(oof_best, 'health_class').round(3))
print('Error by condition (seen before vs brand new):')
display(ev.breakdown(oof_best, 'regime').round(3))
print('Error for each battery (worst 8):')
display(ev.breakdown(oof_best, 'cell_id').sort_values('MAE', ascending=False).head(8).round(3))

My better model on the main test: HistGradientBoosting (Random forest=4.651, HistGradientBoosting=4.275)

Error by health group:


,health_class,n,MAE,RMSE,bias
0,EOL risk (<80),53,13.364,16.601,8.572
1,Degraded (80-90),66,3.378,5.271,-0.782
2,Healthy (>=90),187,2.016,3.100,-1.363


Error by condition (seen before vs brand new):


,regime,n,MAE,RMSE,bias
0,extrapolation,66,10.739,14.763,5.444
1,interpolation,219,2.333,3.590,-1.342
2,unclassified,21,4.212,7.004,3.917


Error for each battery (worst 8):


,cell_id,n,MAE,RMSE,bias
2,2_012,22,15.032,17.945,14.522
10,2_095,16,11.249,17.138,6.711
8,2_076,6,10.877,13.350,10.875
5,2_033,22,7.354,8.494,-7.354
9,2_092,22,6.038,8.344,-6.038
7,2_067,21,4.212,7.004,3.917
13,2_104,22,3.208,4.609,-3.205
12,2_101,22,2.622,3.236,0.225


## 6. Is my better model really better?

A slightly lower average is not enough, because it could just be luck. So I use a proper test (called Wilcoxon) to compare my two models head to head. It asks: does one model really win battery after battery, not just on average? If mean_diff is below zero, Random Forest is the better one. A small p value (under 0.05) means the difference is real. A large p value means the two models are basically tied.

In [7]:
# I compare my two models head to head on the same batteries.
rf = results[('Random forest', 'LOCO')]
hgb = results[('HistGradientBoosting', 'LOCO')]
cmp = pd.DataFrame([ev.paired_comparison(rf, hgb)])
print('Random forest vs HistGradientBoosting on the same batteries:')
print('(mean_diff below 0 means Random forest wins. A small wilcoxon_p means the gap is real, not luck.)')
display(cmp.round(4))

Random forest vs HistGradientBoosting on the same batteries:
(mean_diff below 0 means Random forest wins. A small wilcoxon_p means the gap is real, not luck.)


,a,b,scheme,n_folds,mean_diff,a_wins,b_wins,wilcoxon_p
0,Random forest,HistGradientBoosting,LOCO,15,0.4097,8,7,0.9341


## 7. Which settings did my model pick?

For each hidden battery, my model chose its own best settings. If it keeps picking the same ones, my model is stable and I can trust it. If the choices jump around a lot, that is a sign my dataset is just small, which is something I say honestly in the report.

In [8]:
# I look at the settings my better model chose for each hidden battery.
folds_best = results[(best_name, 'LOCO')].folds[['fold', 'held_out', 'n_test', 'MAE', 'n_selected', 'best_params']]
print(f'Settings my {best_name} chose for each hidden battery:')
display(folds_best.round(3))

Settings my HistGradientBoosting chose for each hidden battery:


,fold,held_out,n_test,MAE,n_selected,best_params
0,0,2_009,21,1.913,25,"{'learning_rate': 0.05, 'max_leaf_nodes': 15, ..."
1,1,2_010,21,2.430,24,"{'learning_rate': 0.05, 'max_leaf_nodes': 7, '..."
2,2,2_012,22,15.032,22,"{'learning_rate': 0.05, 'max_leaf_nodes': 7, '..."
3,3,2_027,22,1.245,24,"{'learning_rate': 0.1, 'max_leaf_nodes': 7, 'm..."
4,4,2_030,22,1.082,24,"{'learning_rate': 0.05, 'max_leaf_nodes': 7, '..."
5,5,2_033,22,7.354,24,"{'learning_rate': 0.05, 'max_leaf_nodes': 15, ..."
6,6,2_058,22,1.070,23,"{'learning_rate': 0.05, 'max_leaf_nodes': 7, '..."
7,7,2_067,21,4.212,24,"{'learning_rate': 0.05, 'max_leaf_nodes': 15, ..."
8,8,2_076,6,10.877,23,"{'learning_rate': 0.1, 'max_leaf_nodes': 15, '..."
9,9,2_092,22,6.038,24,"{'learning_rate': 0.05, 'max_leaf_nodes': 15, ..."


## 8. My final honesty check

At the start of the project, 3 batteries were locked away and never touched. Here I test my final model on them one time only. This is just a safety check. I never use it to choose my model, because that would defeat the purpose. One note: these 3 batteries never got badly worn out, so this is an easy test.

In [9]:
# I run my final model one time on the 3 locked-away batteries.
ho = ev.holdout_check(df, best_spec)
print('Final check on the 3 locked-away batteries:')
for k, v in ho.items():
    print(f'  {k}: {v}')

Final check on the 3 locked-away batteries:
  model: HistGradientBoosting
  n_dev_cells: 12
  n_test: 65
  test_cells: 2_009, 2_092, 2_098
  MAE: 2.942557599460262
  RMSE: 6.021000165803079


## 9. My charts

I draw two charts. The first shows the error of each model under each test method, side by side. The second shows predicted versus actual SOH for my better model. Points sitting on the diagonal line are perfect guesses, and points far from it are mistakes.

In [10]:
# I draw and save my two charts.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Chart 1: error of each model under each test method.
piv = summary.pivot(index='model', columns='scheme', values='MAE')
piv = piv.reindex([x for x in ['Random forest', 'HistGradientBoosting'] if x in piv.index])
piv = piv[[s for s in ['Random 5-fold', 'LOCO', 'LOPO'] if s in piv.columns]]
ax = piv.plot(kind='bar', figsize=(8, 5))
ax.set_ylabel('Error (SOH percent points)')
ax.set_title('Error by model and test method (lower is better)')
ax.set_xlabel('')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'mae_by_model_and_split.png', dpi=130)
plt.show()

# Chart 2: predicted vs actual SOH for my better model.
fig, ax = plt.subplots(figsize=(6, 6))
for reg, g in oof_best.groupby('regime'):
    ax.scatter(g.y_true, g.y_pred, s=18, alpha=0.7, label=reg)
lims = [oof_best[['y_true', 'y_pred']].min().min() - 2, oof_best[['y_true', 'y_pred']].max().max() + 2]
ax.plot(lims, lims, 'k--', lw=1)
ax.set_xlabel('Actual SOH (percent)')
ax.set_ylabel('Predicted SOH (percent)')
ax.set_title(f'{best_name}: predicted vs actual')
ax.legend()
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'pred_vs_actual_loco.png', dpi=130)
plt.show()
print('I saved my charts to', OUT / 'figures')

I saved my charts to /Users/kshitijarohandeshmukh/Downloads/Fall2026/Data245_MachineLearning/project soh/battery-soh-prediction/results/modelling_random_forest_hgb/figures


## 10. I save everything

I save all my tables, predictions, and charts to the results/modelling_rf_hgb folder, so a teammate can use my models in the combined comparison without running anything again.

In [11]:
# I save all my results so the team can use them.
all_oof = pd.concat([r.oof for r in all_results], ignore_index=True)
all_folds = pd.concat([r.folds for r in all_results], ignore_index=True)

all_oof.to_csv(OUT / 'oof_rf_hgb.csv', index=False)
all_folds.to_csv(OUT / 'per_fold_rf_hgb.csv', index=False)
summary.to_csv(OUT / 'metrics_rf_hgb.csv', index=False)
ev.breakdown(oof_best, 'health_class').to_csv(OUT / 'breakdown_health_class.csv', index=False)
ev.breakdown(oof_best, 'regime').to_csv(OUT / 'breakdown_regime.csv', index=False)
ev.breakdown(oof_best, 'cell_id').to_csv(OUT / 'breakdown_per_cell.csv', index=False)
cmp.to_csv(OUT / 'paired_comparison_loco.csv', index=False)
pd.DataFrame([ho]).to_csv(OUT / 'holdout_check.csv', index=False)

print('I saved all my results to', OUT)
for p in sorted(OUT.glob('*.csv')):
    print(' ', p.name)

I saved all my results to /Users/kshitijarohandeshmukh/Downloads/Fall2026/Data245_MachineLearning/project soh/battery-soh-prediction/results/modelling_random_forest_hgb
  breakdown_health_class.csv
  breakdown_per_cell.csv
  breakdown_regime.csv
  holdout_check.csv
  metrics_rf_hgb.csv
  oof_rf_hgb.csv
  paired_comparison_loco.csv
  per_fold_rf_hgb.csv


## My summary in simple words

1. How good are my models on a brand new battery (the real test)? HistGradientBoosting is off by about 4.3 percent on average, and Random Forest by about 4.7 percent. For comparison, just guessing the average battery would be off by about 8.2 percent (that average guess is Samina's baseline), so both of my models clearly help.

2. Which model is better? They are basically tied. HistGradientBoosting is a tiny bit ahead, but the head to head test says that small gap could just be luck (p = 0.93). So I treat them as equally good, with a slight preference for HistGradientBoosting.

3. Why are my numbers bigger than the old notebook 04 (about 2 percent)? Because I used the team's strict and fair testing recipe. The old notebook used an easier setup. My number is the honest one. For comparison, the project's Ridge model scores 4.76 on this same recipe, so my 4.27 is a little better.

4. When does my model struggle? On batteries used in conditions I have never seen (error about 10.7 percent) and on batteries near the end of life (error about 13.4 percent). In those cases it tends to be too optimistic, meaning it thinks the battery is healthier than it really is. This is worth pointing out, because that is exactly where I would most want it to be right.

5. Final check: about 2.9 percent error on the 3 locked-away batteries. But that is an easier set, so I treat it only as a safety check.

Bottom line: both of my tree models work well on unseen batteries, and Random Forest and HistGradientBoosting are basically tied. They can be trusted for healthy and medium aged batteries, but they are less reliable for very worn out batteries or brand new operating conditions, and in those cases they lean optimistic.

To run this notebook again: jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=soh-venv notebooks/06_modelling_random_forest_hgb.ipynb